## Section 0: Imports & Setup
Setup imports, PCA implementation, and output directories.

This notebook is **self-contained** — the `PCAScratch` and `PCAScratchSVD` classes are defined inline below, so no external `.py` files are needed. This makes it ready to run on **Kaggle** or any Jupyter environment.

**To run on Kaggle:**
1. Create a new Kaggle notebook
2. Add the "MNIST" dataset (search for `mnist` in Kaggle datasets)
3. Upload this notebook and click "Run All"


In [ ]:
%matplotlib inline
import os
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, load_wine, load_iris
from sklearn.decomposition import PCA
SklearnPCA = PCA
from sklearn.datasets import fetch_openml, fetch_olivetti_faces
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# Output directories
os.makedirs('figures', exist_ok=True)
os.makedirs('results', exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})
print("Imports done. Output directories created.")

# ============================================================
# FROM-SCRATCH PCA IMPLEMENTATION (inlined from pca_scratch.py)
# ============================================================

"""
From-scratch PCA implementation using eigendecomposition and SVD.

Two classes with unified API:
  - PCAScratch:    eigendecomposition of the covariance matrix (via np.linalg.eigh)
  - PCAScratchSVD: SVD of the centered data matrix (via np.linalg.svd)

Both produce identical results (up to eigenvector sign) and are validated
against sklearn.decomposition.PCA in test_pca.py.

Author: PCA Mini Project
"""

import numpy as np


class PCAScratch:
    """
    Principal Component Analysis via eigendecomposition of the covariance matrix.

    Parameters
    ----------
    n_components : int or None
        Number of principal components to keep. If None and var_threshold
        is also None, all components are kept.
    var_threshold : float or None
        Minimum cumulative explained variance ratio (e.g. 0.95).
        Overrides n_components if both are set.
    """

    def __init__(self, n_components=None, var_threshold=None):
        self.n_components = n_components
        self.var_threshold = var_threshold

    def fit(self, X):
        """
        Fit PCA on training data X.

        1. Compute mean vector μ
        2. Center data: X̃ = X − μ
        3. Covariance matrix: C = X̃ᵀX̃ / (m−1)
        4. Eigendecompose C (symmetric → eigh)
        5. Sort eigenvalues descending; select top-k components

        Parameters
        ----------
        X : ndarray of shape (m, n)
            Training data matrix (m samples, n features).

        Returns
        -------
        self
        """
        X = np.asarray(X, dtype=np.float64)
        m, n = X.shape

        # Step 1–2: Mean and centering
        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_

        # Step 3: Sample covariance matrix  C ∈ ℝ^(n×n)
        C = (Xc.T @ Xc) / (m - 1)

        # Step 4: Eigendecomposition (eigh returns ascending order, real values)
        eigvals, eigvecs = np.linalg.eigh(C)

        # Step 5: Sort descending
        idx = np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]

        # Clip tiny negative eigenvalues (numerical noise) to zero
        eigvals = np.clip(eigvals, 0, None)

        # Store full spectrum
        self.eigvals_all_ = eigvals
        self.eigvecs_all_ = eigvecs  # columns are eigenvectors

        # Explained variance ratios
        total_var = eigvals.sum()
        self.evr_all_ = eigvals / total_var if total_var > 0 else np.zeros_like(eigvals)
        self.cum_evr_ = np.cumsum(self.evr_all_)

        # Determine number of components k
        if self.var_threshold is not None:
            k = int(np.searchsorted(self.cum_evr_, self.var_threshold) + 1)
            k = min(k, n)
        elif self.n_components is not None:
            k = self.n_components
        else:
            k = n

        self.k_ = k
        self.components_ = eigvecs[:, :k]          # n × k (columns = principal directions)
        self.explained_variance_ = eigvals[:k]
        self.explained_variance_ratio_ = self.evr_all_[:k]

        return self

    def transform(self, X):
        """
        Project X onto the top-k principal components.

        Z_k = (X − μ) Q_k

        Parameters
        ----------
        X : ndarray of shape (m, n)

        Returns
        -------
        Z : ndarray of shape (m, k)
        """
        return (np.asarray(X, dtype=np.float64) - self.mean_) @ self.components_

    def inverse_transform(self, Z):
        """
        Reconstruct from reduced representation.

        X̂ = Z Q_kᵀ + μ

        Parameters
        ----------
        Z : ndarray of shape (m, k)

        Returns
        -------
        X_hat : ndarray of shape (m, n)
        """
        return Z @ self.components_.T + self.mean_

    def fit_transform(self, X):
        """Fit and transform in one step."""
        return self.fit(X).transform(X)


class PCAScratchSVD:
    """
    Principal Component Analysis via SVD of the centered data matrix.

    Avoids forming XᵀX explicitly (better numerical stability).
    Relation: X̃ = UΣVᵀ  ⇒  C = V(Σ²/(m−1))Vᵀ  ⇒  λᵢ = σᵢ²/(m−1)

    Parameters
    ----------
    n_components : int or None
        Number of principal components to keep.
    var_threshold : float or None
        Minimum cumulative explained variance ratio.
    """

    def __init__(self, n_components=None, var_threshold=None):
        self.n_components = n_components
        self.var_threshold = var_threshold

    def fit(self, X):
        """
        Fit PCA using SVD.

        Parameters
        ----------
        X : ndarray of shape (m, n)

        Returns
        -------
        self
        """
        X = np.asarray(X, dtype=np.float64)
        m, n = X.shape

        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_

        # SVD: X̃ = U Σ Vᵀ  (full_matrices=False → economy SVD)
        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)

        # Eigenvalues of covariance: λᵢ = σᵢ² / (m−1)
        eigvals = S ** 2 / (m - 1)
        # Principal directions = rows of Vᵀ = columns of V
        eigvecs = Vt.T

        # Store full spectrum
        self.eigvals_all_ = eigvals
        self.eigvecs_all_ = eigvecs

        total_var = eigvals.sum()
        self.evr_all_ = eigvals / total_var if total_var > 0 else np.zeros_like(eigvals)
        self.cum_evr_ = np.cumsum(self.evr_all_)

        # Determine k
        if self.var_threshold is not None:
            k = int(np.searchsorted(self.cum_evr_, self.var_threshold) + 1)
            k = min(k, min(m, n))
        elif self.n_components is not None:
            k = self.n_components
        else:
            k = min(m, n)

        self.k_ = k
        self.components_ = eigvecs[:, :k]
        self.explained_variance_ = eigvals[:k]
        self.explained_variance_ratio_ = self.evr_all_[:k]

        return self

    def transform(self, X):
        """Project X onto the top-k principal components."""
        return (np.asarray(X, dtype=np.float64) - self.mean_) @ self.components_

    def inverse_transform(self, Z):
        """Reconstruct from reduced representation."""
        return Z @ self.components_.T + self.mean_

    def fit_transform(self, X):
        """Fit and transform in one step."""
        return self.fit(X).transform(X)


print("PCAScratch and PCAScratchSVD classes loaded.")


## Section 1: Hand-Worked Example
Step-by-step PCA calculation on a toy dataset.

In [ ]:
X = np.array([[2, 1], [3, 5], [4, 3], [5, 7]], dtype=float)
print("Original X:")
print(X)

mean_vec = np.mean(X, axis=0)
print("\nMean:", mean_vec)

X_centered = X - mean_vec
print("\nCentered X:")
print(X_centered)

cov_mat = (X_centered.T @ X_centered) / (X.shape[0] - 1)
print("\nCovariance Matrix:\n", cov_mat)

eigvals, eigvecs = np.linalg.eigh(cov_mat)
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]
print("\nEigenvalues:", eigvals)
print("Eigenvectors:\n", eigvecs)

evr = eigvals / np.sum(eigvals)
print("\nExplained Variance Ratio:", evr)

k = 1
components = eigvecs[:, :k]
Z = X_centered @ components
print("\nProjected Z (k=1):\n", Z)

X_reconstructed = Z @ components.T + mean_vec
print("\nReconstructed X (k=1):\n", X_reconstructed)

recon_error_sum = np.sum((X - X_reconstructed)**2, axis=1)
recon_error = np.mean(recon_error_sum)
print(f"\nReconstruction Error: {recon_error:.4f}")
print(f"Discarded Eigenvalue: {eigvals[1]:.4f}")

plt.figure(figsize=(6, 6))
plt.scatter(X_centered[:, 0], X_centered[:, 1], c='blue', label='Centered Data')
plt.quiver(0, 0, eigvecs[0, 0], eigvecs[1, 0], color='red', scale=3, label='PC1')
plt.quiver(0, 0, eigvecs[0, 1], eigvecs[1, 1], color='green', scale=3, label='PC2')
for i in range(len(X_centered)):
    proj_point = Z[i, 0] * components[:, 0]
    plt.plot([X_centered[i, 0], proj_point[0]], [X_centered[i, 1], proj_point[1]], 'k--')
plt.title('PCA on 2D Toy Data (4 points)')
plt.axis('equal')
plt.legend()
plt.tight_layout()
plt.savefig('figures/toy_example.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


## Section 2: Validation Against sklearn
Validating `PCAScratch` against `sklearn.decomposition.PCA`.

In [ ]:
digits = load_digits()
X_dig, y_dig = digits.data, digits.target
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_dig, y_dig, test_size=0.2, random_state=42, stratify=y_dig)

k_val = 30
pca_scratch = PCAScratch(n_components=k_val)
pca_scratch.fit(X_train_d)
Z_scratch = pca_scratch.transform(X_train_d)

pca_sk = PCA(n_components=k_val)
pca_sk.fit(X_train_d)
Z_sk = pca_sk.transform(X_train_d)

results = []
eig_match = np.allclose(pca_scratch.eigvals_all_[:k_val], pca_sk.explained_variance_)
results.append(('1. Eigenvalue match', eig_match))

evr_match = np.allclose(pca_scratch.explained_variance_ratio_, pca_sk.explained_variance_ratio_)
results.append(('2. EVR match', evr_match))

dot_prods = np.abs(np.sum(pca_scratch.components_ * pca_sk.components_.T, axis=0))
evec_match = np.allclose(dot_prods, np.ones(k_val))
results.append(('3. Eigenvector match (|dot product| ≈ 1)', evec_match))

Z_match = np.allclose(np.abs(Z_scratch), np.abs(Z_sk))
results.append(('4. Transform match', Z_match))

ortho_match = np.allclose(pca_scratch.components_.T @ pca_scratch.components_, np.eye(k_val), atol=1e-10)
results.append(('5. Orthonormality (Q^T Q = I)', ortho_match))

X_rec_scratch = pca_scratch.inverse_transform(Z_scratch)
X_rec_sk = pca_sk.inverse_transform(Z_sk)
rec_match = np.allclose(X_rec_scratch, X_rec_sk)
results.append(('6. Reconstruction error identity', rec_match))

pca_svd = PCAScratchSVD(n_components=k_val)
pca_svd.fit(X_train_d)
svd_match = np.allclose(pca_scratch.eigvals_all_, pca_svd.eigvals_all_)
results.append(('7. SVD equivalence', svd_match))

Z_cov = np.cov(Z_scratch, rowvar=False)
diag_mask = np.eye(k_val, dtype=bool)
off_diag_match = np.allclose(Z_cov[~diag_mask], 0)
results.append(('8. Decorrelation (cov(Z) = diag(λ))', off_diag_match))

df_val = pd.DataFrame(results, columns=['Test', 'Passed'])
df_val.to_csv('results/validation_tests.csv', index=False)
print("Validation Results:")
print(df_val.to_string(index=False))


## Section 3: MNIST Data Loading
Loading the dataset.

In [ ]:
# Try multiple sources for MNIST: Kaggle dataset, then OpenML, then fallback to digits
dataset_name = None

# 1) Try Kaggle's pre-loaded MNIST dataset
kaggle_mnist_path = '/kaggle/input/mnist-dataset/mnist_784.csv'
kaggle_mnist_path2 = '/kaggle/input/digit-recognizer/train.csv'

if os.path.exists(kaggle_mnist_path):
    print(f"Loading MNIST from Kaggle dataset: {kaggle_mnist_path}")
    df = pd.read_csv(kaggle_mnist_path)
    y_full = df.iloc[:, 0].values.astype(int)
    X_full = df.iloc[:, 1:].values.astype(float) / 255.0
    dataset_name = "MNIST"
elif os.path.exists(kaggle_mnist_path2):
    print(f"Loading MNIST from Kaggle Digit Recognizer: {kaggle_mnist_path2}")
    df = pd.read_csv(kaggle_mnist_path2)
    y_full = df['label'].values.astype(int)
    X_full = df.drop('label', axis=1).values.astype(float) / 255.0
    dataset_name = "MNIST"
else:
    # 2) Try OpenML (needs internet)
    try:
        from sklearn.datasets import fetch_openml
        print("Attempting to load MNIST from OpenML...")
        mnist = fetch_openml('mnist_784', as_frame=False, parser='auto', version=1)
        X_full, y_full = mnist.data, mnist.target.astype(int)
        X_full = X_full / 255.0
        dataset_name = "MNIST"
        print(f"MNIST loaded: {X_full.shape}")
    except Exception as e:
        print(f"MNIST download failed: {e}")

if dataset_name == "MNIST":
    # Standard MNIST split: first 60k train, last 10k test
    X_train, X_test = X_full[:60000], X_full[60000:]
    y_train, y_test = y_full[:60000], y_full[60000:]
    print(f"MNIST loaded successfully!")
else:
    # 3) Fallback to sklearn digits (offline, smaller)
    print("WARNING: MNIST not available. Falling back to sklearn digits (64 features).")
    print("Results will use digits dataset instead.")
    digits = load_digits()
    X_full, y_full = digits.data, digits.target
    X_train, X_test, y_train, y_test = train_test_split(
        X_full, y_full, test_size=0.2, random_state=RANDOM_STATE, stratify=y_full
    )
    dataset_name = "digits"

print(f"\nDataset: {dataset_name}")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Classes: {np.unique(y_train)}")
print(f"Class distribution (train): {np.bincount(y_train)}")


## Section 4: Eigenvalue Spectrum & Cumulative Variance

In [ ]:
print("Fitting PCA on full training set...")
pca_full = PCAScratch()
pca_full.fit(X_train)
print("Done fitting.")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
k_vals = np.arange(1, len(pca_full.eigvals_all_) + 1)

ax1.plot(k_vals, pca_full.eigvals_all_)
ax1.set_xlabel('Component Index')
ax1.set_ylabel('Eigenvalue')
ax1.set_title('Scree Plot (Linear Scale)')

ax2.plot(k_vals, pca_full.eigvals_all_)
ax2.set_xlabel('Component Index')
ax2.set_ylabel('Eigenvalue')
ax2.set_yscale('log')
ax2.set_title('Scree Plot (Log Scale)')

plt.tight_layout()
plt.savefig('figures/scree_plot.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

cum_evr = pca_full.cum_evr_
plt.figure(figsize=(8, 5))
plt.plot(k_vals, cum_evr, label='Cumulative EVR')

thresholds = [0.90, 0.95, 0.99]
colors = ['red', 'green', 'orange']
k_thresholds = {}

for th, c in zip(thresholds, colors):
    k_th = np.argmax(cum_evr >= th) + 1
    k_thresholds[f"{int(th*100)}%"] = k_th
    plt.axhline(y=th, color=c, linestyle='--', label=f'{th*100:.0f}% variance')
    plt.axvline(x=k_th, color=c, linestyle=':', label=f'k={k_th}')

plt.xlabel('Number of Components (k)')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.title('Cumulative Explained Variance vs. k')
plt.legend()
plt.tight_layout()
plt.savefig('figures/cumulative_variance.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

df_th = pd.DataFrame(list(k_thresholds.items()), columns=['Threshold', 'k'])
df_th.to_csv('results/variance_thresholds.csv', index=False)
print("Variance Thresholds:")
print(df_th.to_string(index=False))


## Section 5: Accuracy vs Number of Components

In [ ]:
n_features = X_train.shape[1]
k_values = [2, 5, 10, 20, 30, 50, 100, 200]
k_values = [k for k in k_values if k < n_features] + [n_features]

if dataset_name == "MNIST":
    subset_size = 10000
    np.random.seed(42)
    mnist_sub_idx = np.random.choice(len(X_train), subset_size, replace=False)
    X_train_sub = X_train[mnist_sub_idx]
    y_train_sub = y_train[mnist_sub_idx]
else:
    X_train_sub = X_train
    y_train_sub = y_train

results = []

# Pre-fit PCA for all components once (reuse eigenvalues/vectors)
print("Fitting PCA on full training set (all components)...")
pca_full = PCAScratch()
pca_full.fit(X_train)
Z_train_all = pca_full.transform(X_train)
Z_test_all = pca_full.transform(X_test)

for k in k_values:
    print(f"Evaluating k={k}...")
    if k == n_features:
        Z_tr = X_train_sub
        Z_te = X_test
    else:
        Z_tr = Z_train_all[mnist_sub_idx, :k] if dataset_name == "MNIST" else Z_train_all[:, :k]
        Z_te = Z_test_all[:, :k]

    for clf_name, clf in [
        ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')),
        ('kNN', KNeighborsClassifier(n_neighbors=5)),
        ('SVM', SVC(kernel='rbf', random_state=42)),
    ]:
        # Skip SVM on full dimensions (too slow)
        if clf_name == 'SVM' and k == n_features and n_features > 100:
            print(f"  Skipping SVM on full {n_features}D (too slow)...")
            continue
        
        t0 = time.perf_counter()
        clf.fit(Z_tr, y_train_sub)
        y_pred = clf.predict(Z_te)
        elapsed = time.perf_counter() - t0
        
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')
        results.append({'k': k, 'Classifier': clf_name, 'Accuracy': acc, 'Macro-F1': f1, 'Time (s)': round(elapsed, 2)})
        print(f"  {clf_name} k={k}: acc={acc:.4f} ({elapsed:.1f}s)")

df_acc = pd.DataFrame(results)
df_acc.to_csv('results/accuracy_vs_k.csv', index=False)
print("\nAccuracy vs k:")
print(df_acc.to_string(index=False))

# Plot
plt.figure(figsize=(10, 6))
for clf_name in ['LogisticRegression', 'kNN', 'SVM']:
    sub_df = df_acc[df_acc['Classifier'] == clf_name]
    sub_df_reduced = sub_df[sub_df['k'] < n_features]
    baseline_rows = sub_df[sub_df['k'] == n_features]
    
    plt.plot(sub_df_reduced['k'], sub_df_reduced['Accuracy'], marker='o', label=clf_name)
    if len(baseline_rows) > 0:
        baseline = baseline_rows['Accuracy'].values[0]
        plt.axhline(y=baseline, color=plt.gca().lines[-1].get_color(), linestyle='--', alpha=0.5,
                     label=f'{clf_name} (full {n_features}D)')

plt.xscale('log')
plt.xlabel('Number of Components (k)')
plt.ylabel('Test Accuracy')
plt.title('Accuracy vs Number of PCA Components')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/accuracy_vs_k.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


## Section 6: Timing & Memory

In [ ]:
k_95 = k_thresholds.get('95%', n_features // 2)
print(f"Using k={k_95} (95% variance) for comparison...")

pca_95 = PCAScratch(n_components=k_95)
pca_95.fit(X_train)

if dataset_name == "MNIST":
    timing_subset = 10000
    np.random.seed(42)
    t_idx = np.random.choice(len(X_train), timing_subset, replace=False)
    X_tr_time = X_train[t_idx]
    y_tr_time = y_train[t_idx]
else:
    X_tr_time = X_train
    y_tr_time = y_train

Z_tr_time = pca_95.transform(X_tr_time)
Z_te_time = pca_95.transform(X_test)

timing_results = []

for clf_name, clf_orig, clf_pca in [
    ('LogisticRegression',
     LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs'),
     LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')),
    ('kNN',
     KNeighborsClassifier(n_neighbors=5),
     KNeighborsClassifier(n_neighbors=5)),
]:
    # Original
    t0 = time.perf_counter()
    clf_orig.fit(X_tr_time, y_tr_time)
    fit_time_orig = time.perf_counter() - t0
    
    t0 = time.perf_counter()
    y_pred_orig = clf_orig.predict(X_test)
    pred_time_orig = time.perf_counter() - t0
    acc_orig = accuracy_score(y_test, y_pred_orig)
    
    # PCA-reduced
    t0 = time.perf_counter()
    clf_pca.fit(Z_tr_time, y_tr_time)
    fit_time_pca = time.perf_counter() - t0
    
    t0 = time.perf_counter()
    y_pred_pca = clf_pca.predict(Z_te_time)
    pred_time_pca = time.perf_counter() - t0
    acc_pca = accuracy_score(y_test, y_pred_pca)
    
    timing_results.append({
        'Classifier': clf_name,
        'Original Fit (s)': round(fit_time_orig, 4),
        'PCA Fit (s)': round(fit_time_pca, 4),
        'Original Predict (s)': round(pred_time_orig, 4),
        'PCA Predict (s)': round(pred_time_pca, 4),
        'Fit Speedup': round(fit_time_orig / max(fit_time_pca, 1e-6), 1),
        'Predict Speedup': round(pred_time_orig / max(pred_time_pca, 1e-6), 1),
        'Original Acc': round(acc_orig, 4),
        'PCA Acc': round(acc_pca, 4),
    })

orig_bytes = X_tr_time.nbytes
pca_bytes = Z_tr_time.nbytes
compression = round(orig_bytes / pca_bytes, 1)

print(f"\nMemory: Original={orig_bytes:,} bytes ({n_features}D) → PCA={pca_bytes:,} bytes ({k_95}D)")
print(f"Compression ratio: {compression}×")

df_timing = pd.DataFrame(timing_results)
df_timing.to_csv('results/timing_memory.csv', index=False)
print("\nTiming comparison:")
print(df_timing.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x = np.arange(len(df_timing))
width = 0.35

axes[0].bar(x - width/2, df_timing['Original Fit (s)'], width, label='Original', color='#e74c3c')
axes[0].bar(x + width/2, df_timing['PCA Fit (s)'], width, label=f'PCA (k={k_95})', color='#2ecc71')
axes[0].set_ylabel('Time (s)')
axes[0].set_title('Classifier Fit Time')
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_timing['Classifier'])
axes[0].legend()

axes[1].bar(x - width/2, df_timing['Original Predict (s)'], width, label='Original', color='#e74c3c')
axes[1].bar(x + width/2, df_timing['PCA Predict (s)'], width, label=f'PCA (k={k_95})', color='#2ecc71')
axes[1].set_ylabel('Time (s)')
axes[1].set_title('Classifier Predict Time')
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_timing['Classifier'])
axes[1].legend()

plt.suptitle(f'Original ({n_features}D) vs PCA-Reduced ({k_95}D) — Compression: {compression}×')
plt.tight_layout()
plt.savefig('figures/timing_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


## Section 7: Reconstruction Gallery

In [ ]:
k_gallery = [5, 20, 50, 150]
k_gallery = [k for k in k_gallery if k < n_features]

np.random.seed(42)
sample_idx = []
classes = np.unique(y_test)
for c in classes:
    idx = np.where(y_test == c)[0]
    if len(idx) > 0:
        sample_idx.append(np.random.choice(idx))
sample_idx = sample_idx[:10]

fig, axes = plt.subplots(len(sample_idx), len(k_gallery) + 1, figsize=(2*(len(k_gallery)+1), 2*len(sample_idx)))
img_shape = (28, 28) if dataset_name == "MNIST" else (8, 8)

for row_idx, idx in enumerate(sample_idx):
    orig_img = X_test[idx]
    ax = axes[row_idx, 0]
    ax.imshow(orig_img.reshape(img_shape), cmap='gray')
    if row_idx == 0: ax.set_title('Original')
    ax.axis('off')
    
    for col_idx, k in enumerate(k_gallery):
        pca_k = PCAScratch(n_components=k)
        pca_k.fit(X_train)
        Z_img = pca_k.transform(orig_img.reshape(1, -1))
        rec_img = pca_k.inverse_transform(Z_img)[0]
        mse = np.mean((orig_img - rec_img)**2)
        
        ax = axes[row_idx, col_idx + 1]
        ax.imshow(rec_img.reshape(img_shape), cmap='gray')
        if row_idx == 0: ax.set_title(f'k={k}')
        ax.text(0.5, -0.15, f'MSE: {mse:.4f}', size=9, ha='center', transform=ax.transAxes)
        ax.axis('off')

plt.tight_layout()
plt.savefig('figures/reconstruction_gallery.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


## Section 8: Reconstruction Error vs k

In [ ]:
emp_mse = []
theo_mse = []
k_range = np.arange(1, min(100, n_features) + 1) if dataset_name == "MNIST" else np.arange(1, n_features + 1)

# we can just use the pre-computed eigenvalues for theoretical
total_var = np.sum(pca_full.eigvals_all_)

for k in k_range:
    pca_k = PCAScratch(n_components=k)
    pca_k.fit(X_train)
    Z_tr = pca_k.transform(X_train)
    X_rec = pca_k.inverse_transform(Z_tr)
    
    mse = np.mean(np.sum((X_train - X_rec)**2, axis=1))
    emp_mse.append(mse)
    
    # discarded eigenvalue sum
    disc_var = np.sum(pca_full.eigvals_all_[k:])
    theo_mse.append(disc_var)

plt.figure(figsize=(8, 5))
plt.plot(k_range, emp_mse, label='Empirical MSE', marker='o', alpha=0.7)
plt.plot(k_range, theo_mse, label='Theoretical (Discarded Variance)', linestyle='--', alpha=0.7)
plt.xlabel('k')
plt.ylabel('Reconstruction Error / Variance')
plt.title('Reconstruction Error vs k')
plt.legend()
plt.tight_layout()
plt.savefig('figures/reconstruction_error_vs_k.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


## Section 9: 2D/3D Scatter Projection

In [ ]:
pca_2 = PCAScratch(n_components=2)
pca_2.fit(X_train)
Z_2 = pca_2.transform(X_test)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(Z_2[:, 0], Z_2[:, 1], c=y_test, cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter, ticks=range(10), label='Digit Class')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('2D PCA Projection of Test Data')
plt.tight_layout()
plt.savefig('figures/2d_projection.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


## Section 10: Eigendigits

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
img_shape = (28, 28) if dataset_name == "MNIST" else (8, 8)

for i, ax in enumerate(axes.flat):
    if i < len(pca_full.components_.T):
        eigen_img = pca_full.components_.T[i].reshape(img_shape)
        ax.imshow(eigen_img, cmap='gray')
        ax.set_title(f'PC{i+1}')
    ax.axis('off')

plt.tight_layout()
plt.savefig('figures/eigendigits.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


## Section 11: Eigenfaces

In [ ]:
try:
    from sklearn.datasets import fetch_olivetti_faces
    faces = fetch_olivetti_faces()
    X_faces = faces.data
    
    pca_f = PCAScratch()
    pca_f.fit(X_faces)
    
    mean_face = pca_f.mean_
    
    # Mean face + top 16 eigenfaces
    fig, axes = plt.subplots(4, 5, figsize=(15, 12))
    
    # Mean face in position (0,0)
    axes[0, 0].imshow(mean_face.reshape(64, 64), cmap='gray')
    axes[0, 0].set_title('Mean Face')
    axes[0, 0].axis('off')
    
    # Top 19 eigenfaces
    for i in range(19):
        row = (i + 1) // 5
        col = (i + 1) % 5
        axes[row, col].imshow(pca_f.components_[:, i].reshape(64, 64), cmap='gray')
        axes[row, col].set_title(f'PC{i+1}')
        axes[row, col].axis('off')
    
    plt.suptitle('Mean Face and Top Eigenfaces (Olivetti)', fontsize=14)
    plt.tight_layout()
    plt.savefig('figures/eigenfaces.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    # Face reconstruction at various k
    test_face_idx = 0
    test_face = X_faces[test_face_idx:test_face_idx+1]
    k_vals_face = [5, 20, 50, 100, 200, X_faces.shape[1]]
    k_vals_face = [k for k in k_vals_face if k <= X_faces.shape[1]]
    
    fig, axes = plt.subplots(1, len(k_vals_face) + 1, figsize=(3 * (len(k_vals_face) + 1), 3))
    axes[0].imshow(test_face.reshape(64, 64), cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    for j, k in enumerate(k_vals_face):
        pca_fk = PCAScratch(n_components=k)
        pca_fk.fit(X_faces)
        recon = pca_fk.inverse_transform(pca_fk.transform(test_face))
        mse = np.mean((test_face - recon) ** 2)
        axes[j + 1].imshow(recon.reshape(64, 64), cmap='gray')
        axes[j + 1].set_title(f'k={k}\nMSE={mse:.4f}')
        axes[j + 1].axis('off')
    
    plt.suptitle('Face Reconstruction at Various k', fontsize=14)
    plt.tight_layout()
    plt.savefig('figures/face_reconstruction.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    print("Eigenfaces and face reconstruction complete.")

except Exception as e:
    print(f"Olivetti faces not available: {e}")
    print("Skipping eigenfaces section.")


## Section 12: Standardization Effect (Wine Dataset)
Demonstrates why features with wildly different scales/units require standardization before PCA.

In [ ]:
wine = load_wine()
X_wine = wine.data

pca_raw = PCAScratch()
pca_raw.fit(X_wine)
evr_raw = pca_raw.explained_variance_ratio_

scaler = StandardScaler()
X_wine_std = scaler.fit_transform(X_wine)
pca_std = PCAScratch()
pca_std.fit(X_wine_std)
evr_std = pca_std.explained_variance_ratio_

plt.figure(figsize=(8, 5))
x = np.arange(1, 6)
width = 0.35
plt.bar(x - width/2, evr_raw[:5], width, label='Raw Data (Covariance)', color='#e74c3c')
plt.bar(x + width/2, evr_std[:5], width, label='Standardized (Correlation)', color='#3498db')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Effect of Standardization on EVR (Wine Dataset)')
plt.xticks(x)
plt.legend()
plt.tight_layout()
plt.savefig('figures/standardization_effect.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

max_loading_idx = np.argmax(np.abs(pca_raw.components_[:, 0]))
dom_feat = wine.feature_names[max_loading_idx]
print(f"Feature dominating PC1 in raw data: {dom_feat} (Variance: {np.var(X_wine[:, max_loading_idx]):.1f})")
print(f"PC1 explains {evr_raw[0]*100:.2f}% of raw variance, but only {evr_std[0]*100:.2f}% of standardized variance.")

df_std = pd.DataFrame({'PC': x, 'Raw_EVR': evr_raw[:5], 'Std_EVR': evr_std[:5]})
df_std.to_csv('results/standardization_comparison.csv', index=False)


## Section 13: Eigendecomp vs SVD Timing

In [ ]:
t0 = time.perf_counter()
pca_cov = PCAScratch()
pca_cov.fit(X_train_d)
t_cov = time.perf_counter() - t0

t0 = time.perf_counter()
pca_svd = PCAScratchSVD()
pca_svd.fit(X_train_d)
t_svd = time.perf_counter() - t0

eig_diff = np.max(np.abs(pca_cov.eigvals_all_ - pca_svd.eigvals_all_))

print(f"Eigh Time: {t_cov:.4f} s")
print(f"SVD Time:  {t_svd:.4f} s")
print(f"Max Eigenvalue Difference: {eig_diff:.4e}")

df_svd = pd.DataFrame([{'Method': 'Eigh', 'Time': t_cov}, {'Method': 'SVD', 'Time': t_svd}])
df_svd.to_csv('results/eigh_vs_svd.csv', index=False)


## Section 14: Bonus - Noise Robustness / PCA Denoising

In [ ]:
noise_level = 0.5
X_test_noisy = X_test + np.random.normal(0, noise_level, X_test.shape)
X_test_noisy = np.clip(X_test_noisy, 0, 1)

idx = 0
orig_img = X_test[idx]
noisy_img = X_test_noisy[idx]

k_denoise = [20, 50, 100]
k_denoise = [k for k in k_denoise if k < n_features]

fig, axes = plt.subplots(1, 2 + len(k_denoise), figsize=(12, 3))
img_shape = (28, 28) if dataset_name == "MNIST" else (8, 8)

axes[0].imshow(orig_img.reshape(img_shape), cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(noisy_img.reshape(img_shape), cmap='gray')
axes[1].set_title('Noisy')
axes[1].axis('off')

for i, k in enumerate(k_denoise):
    pca_k = PCAScratch(n_components=k)
    pca_k.fit(X_train)
    Z_noisy = pca_k.transform(noisy_img.reshape(1, -1))
    denoised_img = pca_k.inverse_transform(Z_noisy)[0]
    
    axes[i+2].imshow(denoised_img.reshape(img_shape), cmap='gray')
    axes[i+2].set_title(f'Denoised k={k}')
    axes[i+2].axis('off')

plt.tight_layout()
plt.savefig('figures/denoising.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


## Section 15: Bonus - PCA is Unsupervised

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_iris_std = StandardScaler().fit_transform(X_iris)

pca_iris = PCAScratch()
pca_iris.fit(X_iris_std)
Z_iris = pca_iris.transform(X_iris_std)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(Z_iris[:, 0], Z_iris[:, 1], c=y_iris, cmap='viridis')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('Iris Projected on PC1 and PC2')

axes[1].scatter(Z_iris[:, 1], Z_iris[:, 2], c=y_iris, cmap='viridis')
axes[1].set_xlabel('PC2')
axes[1].set_ylabel('PC3')
axes[1].set_title('Iris Projected on PC2 and PC3')

plt.tight_layout()
plt.savefig('figures/unsupervised_limitation.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


## Section 16: Confusion Matrix for Best Config

In [ ]:
# Find the best config from reduced dimensions only (excluding full-dim SVM which may be skipped)
df_reduced = df_acc[df_acc['k'] < n_features]
best_row = df_reduced.loc[df_reduced['Accuracy'].idxmax()]
best_k = int(best_row['k'])
best_clf_name = best_row['Classifier']
print(f"Best Configuration: {best_clf_name} with k={best_k} (Acc: {best_row['Accuracy']:.4f})")

# Prepare data for refitting the best classifier
if best_k == n_features:
    Z_train_best = X_train_sub
    Z_test_best = X_test
else:
    if dataset_name == "MNIST":
        Z_train_best = Z_train_all[mnist_sub_idx, :best_k]
    else:
        Z_train_best = Z_train_all[:, :best_k]
    Z_test_best = Z_test_all[:, :best_k]

# Safeguard shape check
Z_train_best = np.atleast_2d(Z_train_best)
Z_test_best = np.atleast_2d(Z_test_best)

if best_clf_name == 'LogisticRegression':
    clf_best = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
elif best_clf_name == 'kNN':
    clf_best = KNeighborsClassifier(n_neighbors=5)
else:
    clf_best = SVC(kernel='rbf', random_state=42)

clf_best.fit(Z_train_best, y_train_sub)
y_pred_best = clf_best.predict(Z_test_best)

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, cmap='Blues')
ax.set_title(f'Confusion Matrix: {best_clf_name} (k={best_k})')
plt.tight_layout()
plt.savefig('figures/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


## Section 17: Summary & Conclusion
Summary of findings.

In [ ]:
# === Summary and Conclusion ===
print("=" * 70)
print("SUMMARY AND CONCLUSION")
print("=" * 70)

# Get threshold values
for t in ['90%', '95%', '99%']:
    kv = k_thresholds.get(t, 'N/A')
    print(f"  {t} variance retained at k = {kv} of {n_features} dimensions")

print()

# Best reduced-dimension accuracy
if len(df_acc) > 0:
    df_reduced = df_acc[df_acc['k'] < n_features]
    if len(df_reduced) > 0:
        best = df_reduced.loc[df_reduced['Accuracy'].idxmax()]
        print(f"Best reduced-dim accuracy: {best['Accuracy']:.4f} ({best['Classifier']}, k={int(best['k'])})")
    
    # Full dimension baselines
    df_full = df_acc[df_acc['k'] == n_features]
    if len(df_full) > 0:
        for _, row in df_full.iterrows():
            print(f"Full-dim baseline ({row['Classifier']}): {row['Accuracy']:.4f}")

print()
k95 = k_thresholds.get('95%', 'N/A')
compression = round(n_features / k95, 1) if isinstance(k95, (int, float)) else 'N/A'
print(f"Conclusion: Retaining 95% of variance required k = {k95} of n = {n_features} dimensions")
print(f"           ({compression}× compression)")
print()
print("Key findings:")
print("  1. PCA effectively reduces dimensionality with minimal accuracy loss")
print("  2. From-scratch implementation matches sklearn to numerical precision") 
print("  3. kNN benefits most from PCA (distance computation scales with dimension)")
print("  4. Reconstruction error equals sum of discarded eigenvalues (verified)")
print("=" * 70)
